# Laboratório — processos e threads na prática
**UC3 · SA2 · Aula 2 — "Rodando o próprio caso"**

Neste laboratório você vai criar processos e threads de propósito e observar, no **gerenciador de tarefas**, o que acontece com a CPU, a memória e a GPU.

## Antes de começar
1. Abra o gerenciador de tarefas (**Ctrl + Shift + Esc**) e deixe-o ao lado deste notebook.
2. Na aba **Detalhes**, mostre as colunas **PID**, **Status**, **Threads** e **Memória** (botão direito no cabeçalho → *Selecionar colunas*).
3. Execute a célula de **preparação** e depois um experimento de cada vez (**Shift + Enter**).

Todos os experimentos **terminam sozinhos** em poucos segundos e usam pouca memória. Nenhum deles trava a máquina.

| Experimento | O que o notebook faz | Observe |
|---|---|---|
| A | 1 processo calculando sem parar | Quantos núcleos trabalham? |
| B | Vários processos calculando | Quantos `python` aparecem? |
| C | 1 processo com várias threads | A coluna Threads |
| D | Processos em estados diferentes | CPU de quem calcula × de quem espera |
| E | Processos ocupando memória | A memória total do sistema |
| F | Cálculo na GPU (se houver) | O processo no monitor de GPU |
| G | Muitos processos, cada um com seus dados | Como a memória cresce |

## ⚙️ Preparação (execute esta célula primeiro)

In [ ]:
import os, sys, time, subprocess, threading, shutil

try:
    import psutil
except ImportError:                      # instala a biblioteca psutil, se faltar
    subprocess.check_call([sys.executable, "-m", "pip", "install", "psutil"])
    import psutil

# Programas pequenos que serão executados como PROCESSOS separados
CALCULAR = "import time\nfim = time.time() + {s}\nx = 0\nwhile time.time() < fim:\n    x += 1"
DORMIR   = "import time\ntime.sleep({s})"
GUARDAR  = "import time\ndados = bytearray(b'x') * ({mb} * 1024 * 1024)\ntime.sleep({s})"


def novo_processo(programa, **valores):
    """Cria um processo python separado executando o programa indicado."""
    p = subprocess.Popen([sys.executable, "-c", programa.format(**valores)])
    return psutil.Process(p.pid), p


def encerrar(processos):
    """Garante que os processos criados terminem."""
    for ps, p in processos:
        try:
            if p.poll() is None:
                p.kill()
            p.wait(timeout=5)
        except Exception:
            pass


def mostrar(processos, titulo=""):
    """Mostra PID, status, uso de CPU, memória e threads de cada processo."""
    for ps, _ in processos:                 # primeira leitura de CPU (referência)
        try: ps.cpu_percent(None)
        except psutil.Error: pass
    time.sleep(1)
    if titulo:
        print(titulo)
    print(f"  {'PID':>7} | {'status':<10} | {'CPU':>6} | {'memória':>9} | threads")
    for ps, _ in processos:
        try:
            print(f"  {ps.pid:>7} | {ps.status():<10} | {ps.cpu_percent(None):5.0f}% | "
                  f"{ps.memory_info().rss / 2**20:6.0f} MB | {ps.num_threads()}")
        except psutil.Error:
            print(f"  {ps.pid:>7} | terminado")


def nucleos():
    """Uso de cada núcleo lógico da CPU durante 1 segundo."""
    return " ".join(f"{u:3.0f}%" for u in psutil.cpu_percent(interval=1, percpu=True))


EU = psutil.Process()        # o processo deste notebook
mem = psutil.virtual_memory()
print(f"Este notebook é o processo PID {EU.pid} ({EU.name()}), com {EU.num_threads()} threads.")
print(f"CPU: {psutil.cpu_count(logical=False)} núcleos físicos, {psutil.cpu_count()} núcleos lógicos (threads de hardware)")
print(f"Memória RAM: {mem.total / 2**30:.1f} GB no total, {mem.percent:.0f}% em uso")
print(f"Processos em execução no sistema: {len(psutil.pids())}")
print("\nProcure o PID deste notebook na aba Detalhes do gerenciador de tarefas.")

## Quem está rodando agora?
Os 8 processos com mais threads no sistema. Compare com a aba **Detalhes** do gerenciador de tarefas, ordenada pela coluna Threads.

In [ ]:
lista = []
for p in psutil.process_iter(["pid", "name", "num_threads", "memory_info", "status"]):
    i = p.info
    if i["num_threads"] and i["memory_info"]:
        lista.append((i["num_threads"], i["pid"], i["name"], i["memory_info"].rss / 2**20, i["status"]))

total_threads = sum(x[0] for x in lista)
print(f"{len(lista)} processos visíveis, {total_threads} threads no total\n")
print(f"{'PID':>7} | {'nome':<28} | threads | {'memória':>9} | status")
for th, pid, nome, mb, st in sorted(lista, reverse=True)[:8]:
    print(f"{pid:>7} | {nome[:28]:<28} | {th:>7} | {mb:6.0f} MB | {st}")

## Experimento A — um processo calculando sem parar
Um único processo fica 12 segundos fazendo contas.

**Observe:** quantos núcleos ficam ocupados? O uso total da CPU chega a 100%?

In [ ]:
print("Núcleos antes  :", nucleos())
procs = [novo_processo(CALCULAR, s=12)]
print(f"Processo criado: PID {procs[0][0].pid}  -> procure-o no gerenciador de tarefas\n")
for _ in range(3):
    print("Núcleos durante:", nucleos())
mostrar(procs, "\nO processo criado:")
encerrar(procs)
print("\nUm processo com uma thread ocupa, no máximo, UM núcleo.")

## Experimento B — vários processos calculando
Agora são vários processos iguais ao do experimento A, um para cada núcleo lógico (no máximo 4).

**Observe:** quantos `python` novos aparecem na lista? O que acontece com a CPU?

✏️ **Experimente:** mude `quantos` para um número **maior** que o de núcleos da máquina. Todos conseguem rodar ao mesmo tempo?

In [ ]:
quantos = min(4, psutil.cpu_count())

procs = [novo_processo(CALCULAR, s=12) for _ in range(quantos)]
print(f"{quantos} processos criados: PIDs {[ps.pid for ps, _ in procs]}\n")
for _ in range(3):
    print("Núcleos:", nucleos(), f"  | CPU total: {psutil.cpu_percent(interval=None):.0f}%")
mostrar(procs, "\nOs processos criados:")
encerrar(procs)
print("\nCada processo é independente: o sistema operacional os distribui pelos núcleos.")

## Experimento C — um processo com várias threads
Desta vez não criamos processos novos. Criamos **8 threads dentro do processo do notebook**; elas ficam esperando por 10 segundos.

**Observe:** na aba Detalhes, encontre o PID do notebook e veja a coluna **Threads** aumentar e depois voltar.

In [ ]:
antes = EU.num_threads()
sinal = threading.Event()

def tarefa():
    sinal.wait()                 # a thread fica EM ESPERA até receber o sinal

threads = [threading.Thread(target=tarefa) for _ in range(8)]
for t in threads:
    t.start()

print(f"PID do notebook: {EU.pid}  (o PID não mudou: continua sendo UM processo)")
print(f"Threads antes : {antes}")
print(f"Threads agora : {EU.num_threads()}   (8 a mais)")
print(f"Uso de CPU do processo: {EU.cpu_percent(interval=1):.0f}%  -> threads em espera não gastam CPU")

time.sleep(9)
sinal.set()                      # libera as threads
for t in threads:
    t.join()
print(f"Threads depois: {EU.num_threads()}")
print("\nThreads vivem DENTRO de um processo e dividem a memória dele.")

## Experimento D — processos em estados diferentes
Três processos: um **calculando**, um **dormindo** (em espera) e um **suspenso** (pausado pelo sistema).

**Observe:** o uso de CPU de cada um. Quem está em espera gasta CPU?

No Windows, a coluna *status* mostra `running` para todo processo ativo; a diferença aparece no **uso de CPU**. O processo suspenso aparece como `stopped` (e como "Suspenso" no gerenciador de tarefas).

In [ ]:
calculando = novo_processo(CALCULAR, s=14)
dormindo   = novo_processo(DORMIR, s=14)
suspenso   = novo_processo(CALCULAR, s=14)
suspenso[0].suspend()                       # o sistema operacional pausa o processo

procs = [calculando, dormindo, suspenso]
print(f"calculando: PID {calculando[0].pid} | dormindo: PID {dormindo[0].pid} | suspenso: PID {suspenso[0].pid}")
mostrar(procs, "\nSituação 1 — um calcula, um dorme, um está suspenso:")

suspenso[0].resume()                        # volta para a fila de prontos
mostrar(procs, "\nSituação 2 — o processo suspenso foi retomado:")

encerrar(procs)
mostrar(procs, "\nSituação 3 — todos encerrados:")
print("\nEm execução: usa CPU.  Em espera ou suspenso: 0% de CPU.  Terminado: sai da lista.")

## Experimento E — processos ocupando memória
Quatro processos, cada um guardando um bloco de dados na memória por 10 segundos.

**Observe:** a memória de cada `python` novo e a memória total do sistema, na aba **Desempenho**.

In [ ]:
livre = psutil.virtual_memory().available / 2**30
mb = 150 if livre > 2 else 40              # proteção: usa menos se houver pouca memória livre

antes = psutil.virtual_memory()
procs = [novo_processo(GUARDAR, mb=mb, s=10) for _ in range(4)]
time.sleep(3)
durante = psutil.virtual_memory()
mostrar(procs, f"4 processos, cada um guardando {mb} MB:")

print(f"\nMemória em uso antes  : {antes.used / 2**30:5.2f} GB ({antes.percent:.0f}%)")
print(f"Memória em uso durante: {durante.used / 2**30:5.2f} GB ({durante.percent:.0f}%)")
encerrar(procs)
time.sleep(1)
depois = psutil.virtual_memory()
print(f"Memória em uso depois : {depois.used / 2**30:5.2f} GB ({depois.percent:.0f}%)")
print("\nQuando o processo termina, o sistema operacional recupera a memória dele.")

## Experimento F — o processo na GPU (se houver)
Se a máquina tiver GPU NVIDIA e PyTorch, o notebook faz contas na GPU por alguns segundos e mostra o monitor de GPU.

**Observe:** qual **PID** aparece usando a GPU? É o PID deste notebook?

In [ ]:
tem_smi = shutil.which("nvidia-smi") is not None
try:
    import torch
    tem_cuda = torch.cuda.is_available()
except Exception:
    tem_cuda = False

if tem_cuda:
    a = torch.randn(4096, 4096, device="cuda")
    fim = time.time() + 6
    while time.time() < fim:                 # 6 segundos de multiplicações de matrizes
        b = a @ a
    torch.cuda.synchronize()
    print(f"Cálculo feito na GPU pelo processo PID {EU.pid}.\n")
else:
    print("PyTorch com GPU não disponível: o cálculo na GPU foi pulado.\n")

if tem_smi:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total",
                          "--format=csv"], capture_output=True, text=True).stdout)
    print("Processos usando a GPU:")
    print(subprocess.run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
                          "--format=csv"], capture_output=True, text=True).stdout)
else:
    print("Comando nvidia-smi não encontrado nesta máquina.")
    print("Veja a seção GPU na aba Desempenho do gerenciador de tarefas, ou acompanhe a demonstração do instrutor.")

## Experimento G — muitos processos, cada um com os seus dados
Cada processo carrega a **sua própria cópia** de um mesmo conjunto de dados de 50 MB. O experimento repete isso com 1, 2, 4 e 8 processos.

**Observe:** como a memória total cresce quando o número de processos aumenta.

In [ ]:
mb = 50
print(f"Cada processo guarda {mb} MB de dados.\n")
print(f"{'processos':>9} | {'memória somada':>15} | por processo")
for n in (1, 2, 4, 8):
    procs = [novo_processo(GUARDAR, mb=mb, s=6) for _ in range(n)]
    time.sleep(2.5)
    total = 0
    for ps, _ in procs:
        try: total += ps.memory_info().rss / 2**20
        except psutil.Error: pass
    print(f"{n:>9} | {total:12.0f} MB | {total / n:5.0f} MB")
    encerrar(procs)

print("\nProcessos NÃO dividem memória: n processos = n cópias dos dados.")

## Para registrar e comparar com o caso do travamento

1. No experimento B, quantos processos `python` apareceram? E na Evidência 2 do caso?
2. No experimento C, o que mudou: o número de processos ou o de threads?
3. No experimento D, quanto de CPU usa um processo em espera?
4. No experimento G, o que aconteceu com a memória quando o número de processos dobrou?
5. **Faça a conta:** se cada processo guardasse 790 MB, quanto de memória usariam 16 processos? A sua máquina tem essa memória?

Guarde as respostas: elas serão usadas na reconstituição do caso, na próxima aula.